# 08 - Role-gated SDXL: training

Team One Sheep Army. Trains the role gates (`src/rolegate/`) inside frozen SDXL + scribble ControlNet + IP-Adapter, so that SDXL itself decides, per layer, per pixel and per attribute, how much of each input acts.

| Attribute | Governing input | Gates that can carry it |
|---|---|---|
| Shape, placement | Sketch | `sketch_contour`, `sketch_placement` (ControlNet residuals) |
| Object identity | Text | `text_identity` (token group of the object words) |
| Colour, material | Reference, also when the caption names another | `ref_colour`, `ref_texture` (two reference channels); `text_colour`, `text_material` should learn to close on a conflict |

Training data (all with a known correct output): normal triplets, colour conflicts, material conflicts, and hue pairs (object and reference hue-rotated together). Gates are conditioned on the governance network's predicted conflict flags (notebook 05). Details in `ROLE_GATED_PLAN.md`.

Run order: setup -> memory test (2-3 min) -> full training (about 1.5-2 h on a T4; checkpoints every 150 steps on Drive) -> curves -> before/after check.

In [ ]:
# ---- CONFIG + SETUP (same in notebooks 05-09) ----
DRIVE_ROOT = "/content/drive/MyDrive/sketchy_dataset_one_sketch"
ZIP_PATH = f"{DRIVE_ROOT}/pilot_v1.zip"
LOCAL_DATA = "/content/pilot_v1"
GOV_DIR = f"{DRIVE_ROOT}/governance"            # governance.pt (notebook 05) lives here
ROLE_DIR = f"{GOV_DIR}/role_gates_v2"            # role_gates.pt (notebook 08) lives here (v1 run kept in role_gates/)
# The code lives in the repo's src/ folder (cloned below). To use a copy on Drive instead, set REPO_DIR to it.
REPO_URL = "https://github.com/Harshitha-282007/IE643_final.git"
REPO_BRANCH = "governed-pipeline"
REPO_DIR = "/content/repo"

import os, sys, glob, json, zipfile
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir(REPO_DIR):
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q   # restart the session after a pull that changes src/ (modules are cached)
assert os.path.exists(f"{REPO_DIR}/src/train_role_gates.py"), "src/train_role_gates.py not found: push the new code or point REPO_DIR at it"
sys.path.insert(0, f"{REPO_DIR}/src")
os.makedirs(ROLE_DIR, exist_ok=True)
if not glob.glob(f"{LOCAL_DATA}/**/metadata.jsonl", recursive=True):
    os.makedirs(LOCAL_DATA, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(LOCAL_DATA)
DATA_DIR = os.path.dirname(glob.glob(f"{LOCAL_DATA}/**/metadata.jsonl", recursive=True)[0])
BASE_ROWS = [json.loads(l) for l in open(f"{DATA_DIR}/metadata.jsonl") if l.strip()]
GOV = f"{GOV_DIR}/governance.pt"
assert os.path.exists(GOV), "governance.pt missing: run notebook 05 first"
print("data:", DATA_DIR, "| base rows:", len(BASE_ROWS))

In [ ]:
!pip uninstall -y torchao -q
!pip install -q -U diffusers transformers accelerate safetensors peft lpips opencv-python-headless
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from pipeline import Pipeline
import train_role_gates as TR
print(torch.cuda.get_device_name(0), "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## Memory test
One step of every kind (warm-up, attribute loss, swap pair) on a few photos. Must print `MEMORY TEST OK` with peak memory under ~14 GB. The loaded pipeline is reused below.

In [ ]:
RES = 512
P = Pipeline(res=RES)
P, _, _ = TR.train(DATA_DIR, GOV, f"{ROLE_DIR}/memtest", res=RES, memory_test=True, P=P)

## Full training
600 optimizer steps x 4 micro-batches. Checkpoints `role_gates_stepXXXX.pt` every 150 steps and `train_log.json` are written to Drive, so a disconnect loses at most 150 steps.

In [ ]:
P, role, hist = TR.train(DATA_DIR, GOV, ROLE_DIR, res=RES, P=P)
print(open(f"{ROLE_DIR}/role_gates_config.json").read())

In [ ]:
# ---- training curves ----
H = pd.DataFrame(json.load(open(f"{ROLE_DIR}/train_log.json")))
cols = [c for c in ["diff", "attr", "swap", "prior", "keep", "a_colour", "a_texture", "a_contour", "a_identity"] if c in H]
fig, ax = plt.subplots(2, (len(cols) + 1) // 2, figsize=(4 * ((len(cols) + 1) // 2), 6))
for a, c in zip(ax.ravel(), cols):
    s = H.groupby("step")[c].mean().dropna()
    a.plot(s.index, s.values, lw=0.6, alpha=0.4); a.plot(s.index, s.rolling(20, min_periods=1).mean().values, lw=1.5)
    for x in (150, 450): a.axvline(x, color="k", lw=0.5, ls="--")
    a.set_title(c)
plt.tight_layout(); plt.savefig(f"{ROLE_DIR}/train_curves.png", dpi=110); plt.show()
print(H.groupby("phase")[cols].mean().round(4))

## Before / after on held-out colour conflicts
Same item, gates at initialisation (equivalent to the fixed pipeline with InstantStyle) vs trained gates. The trained model should follow the reference colour although the caption names another one; the gate maps show where each input acts.

In [ ]:
from governance import labels as L, model as GM
from rolegate.gates import RoleGates, gate_maps, CROSS_GATES, SKETCH_GATES
MAPS = CROSS_GATES + SKETCH_GATES
from pipeline import seed_for
P.res, P.steps = 640, 20
net = GM.load(GOV)
test = [r for r in L.expand_rows(BASE_ROWS, per_type=1, seed=1, kinds=("col",)) if r["split"] == "test" and r["id"] != r["original_id"]][:3]
op = lambda r, k: Image.open(f"{DATA_DIR}/{r[k]}")
P.prepare_object_dirs([r.get("ref_object") or "object" for r in test])
items = P.make_items([dict(key=r["id"], sketch=op(r, "sketch"), ref=op(r, "reference"), caption=r["caption"], obj=r["category"],
                           seed=seed_for(r["original_id"]), ref_object=r.get("ref_object")) for r in test])
init = RoleGates.for_unet(P.unet, P.controlnet_res_channels())
trained = RoleGates.load(f"{ROLE_DIR}/role_gates.pt", P.dev)
fig, ax = plt.subplots(len(test), 4 + len(MAPS), figsize=(2.6 * (4 + len(MAPS)), 2.8 * len(test)))
for i, (r, it) in enumerate(zip(test, items)):
    gp = GM.predict_gates(net, it["x"]); fl = [gp["g_obj"], gp["g_col"], gp["g_mat"]]
    a, _ = P.run_role(it, init, fl)
    b, rec = P.run_role(it, trained, fl, record_at=10)
    gm = gate_maps(rec)
    for j, (im, t) in enumerate([(it["sketch"], "sketch"), (it["ref"], f"ref: {r['ref_colors']}"), (a, "gates at init"), (b, "trained gates")]):
        ax[i, j].imshow(im); ax[i, j].set_title(t, fontsize=8)
    for j, g in enumerate(MAPS):
        ax[i, 4 + j].imshow(gm[g].numpy(), vmin=0, vmax=2, cmap="magma"); ax[i, 4 + j].set_title(g, fontsize=8)
    ax[i, 0].set_ylabel(r["caption"][:40], fontsize=7)
for x in ax.ravel(): x.set_xticks([]); x.set_yticks([])
plt.tight_layout(); plt.savefig(f"{ROLE_DIR}/before_after.png", dpi=110); plt.show()